# Metacognitive Medical Digital Twins Pipeline

This notebook implements the complete MDT pipeline with MIMIC-IV and Medical-O1 data sources.

**Alignment Components:**
- Theory of Mind module for user belief inference
- Composite reward engine (5 components: safety, empathy, proactivity, metacognition, semantic)
- GRPO training for multi-objective alignment

**Ontology Components:**
- LOINC, SNOMED-CT, ICD-10 code mappings
- Clinical reference ranges validation
- MIMIC-IV item ID mappings

**Data Sources:**
- MIMIC-IV (ICU trajectories)
- Medical-O1 (reasoning chains)


## 1. Setup & Environment

In [1]:
# Environment setup
import sys
sys.path.append(".")

# Core imports
from config.configs import DataConfig
from data.mimic_processor import MIMICProcessor
from data.medical_o1_processor import MedicalO1Processor
from core.theory_of_mind import TheoryOfMindModule
from rewards.composite_engine import CompositeRewardEngine
from training.grpo_trainer import run_grpo_training
from utils.ontology_validator import OntologyValidator
from utils.helpers import clean_memory, setup_logging

print("✓ All components imported successfully")


✓ All components imported successfully


## 2. Configuration

In [2]:
# Setup logging
setup_logging()
import logging
logger = logging.getLogger(__name__)

logger.info("="*80)
logger.info("MEDICAL DIGITAL TWIN - MASTER PIPELINE")
logger.info("="*80)

# Load configuration
data_config = DataConfig()
print(f"✓ Configuration loaded")
print(f"  MIMIC patients: {data_config.max_patients}")
print(f"  Medical-O1 examples: {data_config.max_o1_examples}")


2026-03-26 09:49:27,023 - __main__ - INFO - ================================================================================
2026-03-26 09:49:27,024 - __main__ - INFO - MEDICAL DIGITAL TWIN - MASTER PIPELINE
2026-03-26 09:49:27,025 - __main__ - INFO - ================================================================================
✓ Configuration loaded
  MIMIC patients: 1000
  Medical-O1 examples: 5000
2026-03-26 09:49:27,024 - __main__ - INFO - MEDICAL DIGITAL TWIN - MASTER PIPELINE
2026-03-26 09:49:27,025 - __main__ - INFO - ================================================================================
✓ Configuration loaded
  MIMIC patients: 1000
  Medical-O1 examples: 5000


## 3. Data Loading

In [3]:
# Load MIMIC-IV data
print("Loading MIMIC-IV data...")
mimic_processor = MIMICProcessor(data_config)

if mimic_processor.check_availability():
    mimic_data = mimic_processor.process_all_patients(
        max_patients=data_config.max_patients
    )
    print(f"✓ Loaded {len(mimic_data)} MIMIC examples")
    
    # Load ICD diagnoses for validation
    try:
        icd_diagnoses = mimic_processor.load_icd_diagnoses()
        print(f"✓ Loaded {len(icd_diagnoses)} ICD diagnoses")
    except Exception as e:
        print(f"⚠️ ICD diagnoses loading failed: {e}")
        icd_diagnoses = None
else:
    print("⚠️ MIMIC data not available")
    mimic_data = []
    icd_diagnoses = None

Loading MIMIC-IV data...
2026-03-26 09:49:31,281 - data.mimic_processor - INFO - Initialized MIMIC-IV v3.1 processor
2026-03-26 09:49:31,282 - data.mimic_processor - INFO - Root directory: mimiciv/3.1
2026-03-26 09:49:31,290 - data.mimic_processor - INFO - Loaded 61 LOINC codes from ontology
2026-03-26 09:49:31,291 - data.mimic_processor - INFO - Loaded 19 chartevents item categories
2026-03-26 09:49:31,282 - data.mimic_processor - INFO - Root directory: mimiciv/3.1
2026-03-26 09:49:31,290 - data.mimic_processor - INFO - Loaded 61 LOINC codes from ontology
2026-03-26 09:49:31,291 - data.mimic_processor - INFO - Loaded 19 chartevents item categories
2026-03-26 09:49:31,291 - data.mimic_processor - INFO - Loaded 32 reference ranges
2026-03-26 09:49:31,291 - data.mimic_processor - INFO - Loaded 32 reference ranges
2026-03-26 09:49:31,294 - data.mimic_processor - INFO - All required MIMIC-IV files found
2026-03-26 09:49:31,298 - data.mimic_processor - INFO - Processing up to 1000 patients.

In [5]:
# Load Medical-O1 data
print("Loading Medical-O1 data...")
o1_processor = MedicalO1Processor()

try:
    o1_dataset = o1_processor.load_dataset(
        split='train',
        config=data_config.medical_o1_config
    )
    if o1_dataset:
        o1_data = o1_processor.format_for_training(
            o1_dataset,
            max_examples=data_config.max_o1_examples
        )
        print(f"✓ Loaded {len(o1_data)} Medical-O1 examples")
    else:
        print("⚠️ Medical-O1 dataset loading failed")
        o1_data = []
except Exception as e:
    print(f"⚠️ Medical-O1 loading failed: {e}")
    o1_data = []

# Combine datasets
all_training_data = mimic_data + o1_data
print(f"✓ Total training examples: {len(all_training_data)}")
print(f"  MIMIC-IV: {len(mimic_data)}")
print(f"  Medical-O1: {len(o1_data)}")

Loading Medical-O1 data...
2026-03-26 09:50:01,114 - data.medical_o1_processor - INFO - Initialized MedicalO1Processor
2026-03-26 09:50:01,115 - data.medical_o1_processor - INFO - Loading FreedomIntelligence/medical-o1-reasoning-SFT (config=en) from HuggingFace...
2026-03-26 09:50:01,115 - data.medical_o1_processor - INFO - Loading FreedomIntelligence/medical-o1-reasoning-SFT (config=en) from HuggingFace...
2026-03-26 09:50:01,240 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/FreedomIntelligence/medical-o1-reasoning-SFT/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-03-26 09:50:01,240 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/FreedomIntelligence/medical-o1-reasoning-SFT/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"


2026-03-26 09:50:01,242 - huggingface_hub.utils._http - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-03-26 09:50:01,255 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/FreedomIntelligence/medical-o1-reasoning-SFT/fc2c9e8a37b38f38da6d449564a8c350b244aef4/README.md "HTTP/1.1 200 OK"
2026-03-26 09:50:01,255 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/FreedomIntelligence/medical-o1-reasoning-SFT/fc2c9e8a37b38f38da6d449564a8c350b244aef4/README.md "HTTP/1.1 200 OK"
2026-03-26 09:50:01,292 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/resolve-cache/datasets/FreedomIntelligence/medical-o1-reasoning-SFT/fc2c9e8a37b38f38da6d449564a8c350b244aef4/README.md "HTTP/1.1 200 OK"
2026-03-26 09:50:01,292 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/resolve-cache/datasets/FreedomIntellige

README.md: 0.00B [00:00, ?B/s]

2026-03-26 09:50:01,372 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/FreedomIntelligence/medical-o1-reasoning-SFT/resolve/fc2c9e8a37b38f38da6d449564a8c350b244aef4/medical-o1-reasoning-SFT.py "HTTP/1.1 404 Not Found"
2026-03-26 09:50:01,496 - httpx - INFO - HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/FreedomIntelligence/medical-o1-reasoning-SFT/FreedomIntelligence/medical-o1-reasoning-SFT.py "HTTP/1.1 404 Not Found"
2026-03-26 09:50:01,496 - httpx - INFO - HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/FreedomIntelligence/medical-o1-reasoning-SFT/FreedomIntelligence/medical-o1-reasoning-SFT.py "HTTP/1.1 404 Not Found"
2026-03-26 09:50:01,549 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/datasets/FreedomIntelligence/medical-o1-reasoning-SFT/revision/fc2c9e8a37b38f38da6d449564a8c350b244aef4 "HTTP/1.1 200 OK"
2026-03-26 09:50:01,549 - httpx - INFO - HTTP Request: GET http

medical_o1_sft.json:   0%|          | 0.00/58.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/19704 [00:00<?, ? examples/s]

2026-03-26 09:50:04,905 - data.medical_o1_processor - INFO - ✓ Loaded 19704 examples from Medical-O1 dataset
2026-03-26 09:50:04,906 - data.medical_o1_processor - INFO - Formatting 5000 Medical-O1 examples...
2026-03-26 09:50:04,906 - data.medical_o1_processor - INFO - Formatting 5000 Medical-O1 examples...
2026-03-26 09:50:05,527 - data.medical_o1_processor - INFO - ✓ Formatted 5000 Medical-O1 examples
✓ Loaded 5000 Medical-O1 examples
✓ Total training examples: 5160
  MIMIC-IV: 160
  Medical-O1: 5000
2026-03-26 09:50:05,527 - data.medical_o1_processor - INFO - ✓ Formatted 5000 Medical-O1 examples
✓ Loaded 5000 Medical-O1 examples
✓ Total training examples: 5160
  MIMIC-IV: 160
  Medical-O1: 5000


## 3.5. Ontology Validation

## 4. Model Training (SFT)

In [6]:
# Initialize ontology validator
print("Initializing ontology validator...")
ontology_validator = OntologyValidator()

# Validate ICD codes if available
if icd_diagnoses is not None:
    print("Validating ICD-10 codes...")
    valid_icd_count = 0
    total_icd_count = len(icd_diagnoses)
    
    # Sample validation (check first 100 for speed)
    sample_icd = icd_diagnoses.head(100)
    for _, row in sample_icd.iterrows():
        icd_code = row['icd_code']
        if ontology_validator.validate_icd10_code(icd_code):
            valid_icd_count += 1
    
    print(f"✓ ICD-10 validation: {valid_icd_count}/{len(sample_icd)} sample codes valid")
    print(f"  Total ICD diagnoses available: {total_icd_count}")
else:
    print("⚠️ ICD diagnoses not available for validation")

# Validate LOINC codes from loaded data
print("Validating LOINC codes...")
loinc_codes = ontology_validator.loinc_codes
print(f"✓ Loaded {len(loinc_codes)} LOINC codes in ontology")

# Test some common LOINC codes
test_loinc = ['8867-4', '8480-6', '8462-4', '2524-7', '2160-0']  # HR, SBP, DBP, Lactate, Creatinine
valid_loinc_count = 0
for code in test_loinc:
    if ontology_validator.validate_loinc_code(code):
        valid_loinc_count += 1

print(f"✓ LOINC validation: {valid_loinc_count}/{len(test_loinc)} test codes valid")

# Validate SNOMED-CT codes
print("Validating SNOMED-CT codes...")
snomed_codes = ontology_validator.snomed_codes
print(f"✓ Loaded {len(snomed_codes)} SNOMED-CT codes in ontology")

# Test some common SNOMED codes
test_snomed = ['91302008', '233604007', '67782005', '84114007', '22298006']  # Sepsis, Pneumonia, ARDS, Heart Failure, MI
valid_snomed_count = 0
for code in test_snomed:
    if ontology_validator.validate_snomed_code(code):
        valid_snomed_count += 1

print(f"✓ SNOMED-CT validation: {valid_snomed_count}/{len(test_snomed)} test codes valid")

print("✓ Complete ontology validation completed")

Initializing ontology validator...
2026-03-26 09:50:08,265 - utils.ontology_validator - INFO - Initialized OntologyValidator
2026-03-26 09:50:08,265 - utils.ontology_validator - INFO -   LOINC codes: 61
2026-03-26 09:50:08,266 - utils.ontology_validator - INFO -   SNOMED codes: 50
2026-03-26 09:50:08,266 - utils.ontology_validator - INFO -   ICD-10 codes: 52
2026-03-26 09:50:08,266 - utils.ontology_validator - INFO -   Reference ranges: 32
Validating ICD-10 codes...
✓ ICD-10 validation: 2/100 sample codes valid
  Total ICD diagnoses available: 6364488
Validating LOINC codes...
✓ Loaded 61 LOINC codes in ontology
✓ LOINC validation: 5/5 test codes valid
Validating SNOMED-CT codes...
✓ Loaded 50 SNOMED-CT codes in ontology
✓ SNOMED-CT validation: 5/5 test codes valid
✓ Complete ontology validation completed
2026-03-26 09:50:08,265 - utils.ontology_validator - INFO -   LOINC codes: 61
2026-03-26 09:50:08,266 - utils.ontology_validator - INFO -   SNOMED codes: 50
2026-03-26 09:50:08,266 - 

In [9]:
# Import SFT training components
from training.sft_trainer import run_sft_training
from models.mdt_model import MedicalDigitalTwinModel
from config.configs import ModelConfig, SFTConfig
from data.dataset import CognitiveStreamDataset

# Initialize model configuration
model_config = ModelConfig()
print(f"✓ Model config loaded: {model_config.model_name}")

# Initialize SFT training configuration
sft_config = SFTConfig()
print(f"✓ SFT config loaded: {sft_config.num_epochs} epochs")

# Initialize model
model = MedicalDigitalTwinModel(model_config)

# Prepare training data split
if len(all_training_data) > 0:
    # Split data for training and evaluation (90% train, 10% eval)
    split_idx = int(len(all_training_data) * 0.9)
    train_data = all_training_data[:split_idx]
    eval_data = all_training_data[split_idx:]
    
    # Create dataset objects
    train_dataset = CognitiveStreamDataset(
        train_data,
        model.tokenizer,
        max_length=model_config.max_length
    )
    
    eval_dataset = CognitiveStreamDataset(
        eval_data,
        model.tokenizer,
        max_length=model_config.max_length
    )
    
    print(f"✓ Data split: {len(train_dataset)} train, {len(eval_dataset)} eval examples")
else:
    train_dataset = None
    eval_dataset = None
    print("⚠️ No training data available")

# Run SFT training
print("Starting SFT training...")
if train_dataset is not None and eval_dataset is not None:
    trained_model = run_sft_training(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        config=sft_config
    )
    print("✓ SFT training completed")
else:
    print("⚠️ Skipping SFT training - no data available")
    trained_model = model

✓ Model config loaded: Qwen/Qwen3.5-4B
✓ SFT config loaded: 3 epochs
2026-03-26 09:51:38,563 - models.mdt_model - WARNING - Using GPT-2 for demo. Set use_demo_model=False for production.
2026-03-26 09:51:38,564 - models.mdt_model - INFO - Loading GPT-2 demo model...
2026-03-26 09:51:38,564 - models.mdt_model - INFO - Loading GPT-2 demo model...
2026-03-26 09:51:38,631 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/config.json "HTTP/1.1 200 OK"
2026-03-26 09:51:38,631 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/config.json "HTTP/1.1 200 OK"
2026-03-26 09:51:38,731 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/config.json "HTTP/1.1 200 OK"
2026-03-26 09:51:38,731 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/config.json "HTTP/1.1 200 OK"
2026-03-26 09:51:38,779 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/model.safetensors "HTTP/1.1 

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

2026-03-26 09:51:38,950 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/generation_config.json "HTTP/1.1 200 OK"
2026-03-26 09:51:38,998 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
2026-03-26 09:51:38,998 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
2026-03-26 09:51:39,044 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/gpt2/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-03-26 09:51:39,044 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/gpt2/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-03-26 09:51:39,088 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/openai-community/gpt2/tree/main/additional_chat_templates?recursive=false&expand=false "H

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


2026-03-26 09:51:40,136 - training.sft_trainer - ERROR - Trainer failed: Your setup doesn't support bf16/gpu. You need to assign use_cpu if you want to train the model on CPU.
Traceback (most recent call last):
  File "/blue/prismap-ai-core/Ahmed/DigitalTwins/MDT/training/sft_trainer.py", line 154, in run_sft_training
    training_args = TrainingArguments(**filtered_kwargs)
  File "<string>", line 112, in __init__
  File "/blue/mrouhizadeh/ahmed.soliman/.conda/envs/digitaltwins_env/lib/python3.13/site-packages/transformers/training_args.py", line 1524, in __post_init__
    self._validate_args()
    ~~~~~~~~~~~~~~~~~~~^^
  File "/blue/mrouhizadeh/ahmed.soliman/.conda/envs/digitaltwins_env/lib/python3.13/site-packages/transformers/training_args.py", line 1686, in _validate_args
    raise ValueError(error_message)
ValueError: Your setup doesn't support bf16/gpu. You need to assign use_cpu if you want to train the model on CPU.
2026-03-26 09:51:40,138 - training.sft_trainer - WARNING - Fal

Epoch 1/3:   0%|          | 0/1161 [00:00<?, ?it/s]`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
Epoch 1/3:   2%|▏         | 20/1161 [08:59<8:32:45, 26.96s/it, loss=7.4022, lr=9.30e-07]



KeyboardInterrupt: 

## 5. Alignment Training (GRPO)

In [ ]:
# Import GRPO training components
from training.grpo_trainer import run_grpo_training
from config.configs import GRPOConfig
from data.dataset import CognitiveStreamDataset
from torch.utils.data import DataLoader

# Initialize reward engine
reward_engine = CompositeRewardEngine()

# Create GRPO configuration
grpo_config = GRPOConfig()
print(f"✓ GRPO config loaded: {grpo_config.num_iterations} iterations")

# Create dataloader for GRPO training
if train_dataset is not None:
    # Use the same dataset objects created for SFT training
    grpo_dataloader = DataLoader(
        train_dataset,
        batch_size=grpo_config.batch_size,
        shuffle=True
    )
    print(f"✓ Created GRPO dataloader with {len(train_dataset)} examples")
else:
    grpo_dataloader = None
    print("⚠️ No training data available for GRPO")

# Run GRPO alignment
print("Starting GRPO alignment training...")
if grpo_dataloader is not None:
    aligned_model = run_grpo_training(
        model=trained_model,
        train_dataloader=grpo_dataloader,
        config=grpo_config,
        reward_engine=reward_engine
    )
    print("✓ GRPO alignment completed")
else:
    print("⚠️ Skipping GRPO training - no dataloader available")
    aligned_model = trained_model

## 6. Evaluation

In [ ]:
# Import evaluation
from evaluation.evaluator import MedicalTwinEvaluator

# Run evaluation
evaluator = MedicalTwinEvaluator()
results = evaluator.evaluate_model(aligned_model)

print("✓ Evaluation completed")
print(f"Results: {results}")
